# 01_preproc - Preprocessing MovieLens

### Obiettivo
 Notebook per la costruzione di un dataset strutturato utente-anno basato sui dati MovieLens, con estrazione dei generi cinematografici e calcolo di metriche preliminari di diversità.

Il risultato sarà un dataset pronto per analisi esplorativa e modelli statistici.

### Import librerie

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

print("Ambiente configurato correttamente")

Ambiente configurato correttamente


### Definizione path progetto

In [2]:
cwd = Path.cwd()

if cwd.name == "notebooks":
    project_root = cwd.parent
else:
    project_root = cwd

data_raw = project_root / "data" / "raw" / "ml-25m"
data_processed = project_root / "data" / "processed"

print("Project root:", project_root)

Project root: /Users/sofiadimaggio/Desktop/uni/3 anno/Tirocinio_MovieLens


### Caricamento dataset

In [3]:
ratings = pd.read_csv(data_raw / "ratings.csv")
movies = pd.read_csv(data_raw / "movies.csv")

print("ratings:", ratings.shape)
print("movies:", movies.shape)

ratings: (25000095, 4)
movies: (62423, 3)


In [4]:
ratings.columns

Index(['userId', 'movieId', 'rating', 'timestamp'], dtype='object')

### Pulizia dei generi

In [5]:
movies["genres"] = movies["genres"].fillna("Unknown")
movies = movies[movies["genres"] != "(no genres listed)"]

### Creazione variabile anno

In [6]:
ratings["year"] = pd.to_datetime(ratings["timestamp"], unit="s").dt.year

ratings.head()

,userId,movieId,rating,timestamp,year
0,1,296,5.0,1147880044,2006
1,1,306,3.5,1147868817,2006
2,1,307,5.0,1147868828,2006
3,1,665,5.0,1147878820,2006
4,1,899,3.5,1147868510,2006


### Preparazione generi

In [7]:
movies["genres"] = movies["genres"].str.split("|")

movies.head()

,movieId,title,genres
0,1,Toy Story (1995),"[Adventure, Animation, Children, Comedy, Fantasy]"
1,2,Jumanji (1995),"[Adventure, Children, Fantasy]"
2,3,Grumpier Old Men (1995),"[Comedy, Romance]"
3,4,Waiting to Exhale (1995),"[Comedy, Drama, Romance]"
4,5,Father of the Bride Part II (1995),[Comedy]


### Merge dataset

In [8]:
df = ratings.merge(movies, on="movieId")

df.head()

,userId,movieId,rating,timestamp,year,title,genres
0,1,296,5.0,1147880044,2006,Pulp Fiction (1994),"[Comedy, Crime, Drama, Thriller]"
1,1,306,3.5,1147868817,2006,Three Colors: Red (Trois couleurs: Rouge) (1994),[Drama]
2,1,307,5.0,1147868828,2006,Three Colors: Blue (Trois couleurs: Bleu) (1993),[Drama]
3,1,665,5.0,1147878820,2006,Underground (1995),"[Comedy, Drama, War]"
4,1,899,3.5,1147868510,2006,Singin' in the Rain (1952),"[Comedy, Musical, Romance]"


### Explode generi

In [9]:
df = df.explode("genres")

df.head()

,userId,movieId,rating,timestamp,year,title,genres
0,1,296,5.0,1147880044,2006,Pulp Fiction (1994),Comedy
0,1,296,5.0,1147880044,2006,Pulp Fiction (1994),Crime
0,1,296,5.0,1147880044,2006,Pulp Fiction (1994),Drama
0,1,296,5.0,1147880044,2006,Pulp Fiction (1994),Thriller
1,1,306,3.5,1147868817,2006,Three Colors: Red (Trois couleurs: Rouge) (1994),Drama


### Aggregazione utente-anno-genere

In [10]:
user_year_genre = df.groupby(
    ["userId", "year", "genres"]
).size().reset_index(name="count")

user_year_genre = user_year_genre.sort_values(["userId", "year"])
user_year_genre.head()

,userId,year,genres,count
0,1,2006,Action,4
1,1,2006,Adventure,11
2,1,2006,Animation,2
3,1,2006,Children,3
4,1,2006,Comedy,23


### Calcolo novelty

In [11]:
novelty_results = []

for user_id, group in user_year_genre.groupby("userId"):
    seen_genres = set()

    for year, sub in group.groupby("year"):
        current_genres = set(sub["genres"])

        if len(current_genres) == 0:
            novelty = 0
        else:
            new_genres = current_genres - seen_genres
            novelty = len(new_genres) / len(current_genres)

        novelty_results.append({
            "userId": user_id,
            "year": year,
            "novelty_pct": novelty
        })

        seen_genres.update(current_genres)

user_year_novelty = pd.DataFrame(novelty_results)

### Pivot: matrice utente-tempo

In [12]:
pivot = user_year_genre.pivot_table(
    index=["userId", "year"],
    columns="genres",
    values="count",
    fill_value=0
).reset_index()

pivot.head()

genres,userId,year,Action,Adventure,Animation,Children,Comedy,Crime,Documentary,Drama,...,Film-Noir,Horror,IMAX,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,1,2006,4.0,11.0,2.0,3.0,23.0,8.0,1.0,53.0,...,1.0,1.0,0.0,5.0,4.0,18.0,5.0,5.0,5.0,1.0
1,2,2006,66.0,75.0,17.0,25.0,63.0,18.0,0.0,91.0,...,0.0,3.0,6.0,11.0,8.0,34.0,28.0,30.0,15.0,5.0
2,3,2015,204.0,129.0,29.0,29.0,106.0,67.0,3.0,143.0,...,2.0,33.0,45.0,5.0,36.0,48.0,145.0,153.0,12.0,4.0
3,3,2016,15.0,9.0,2.0,2.0,7.0,6.0,0.0,11.0,...,1.0,0.0,5.0,0.0,2.0,2.0,7.0,10.0,3.0,0.0
4,3,2017,34.0,18.0,6.0,2.0,12.0,14.0,0.0,24.0,...,1.0,5.0,13.0,0.0,10.0,1.0,42.0,25.0,6.0,1.0


### Entropia di Shannon

In [13]:
def shannon_entropy(row):
    values = row.drop(["userId", "year"]).values
    total = values.sum()

    if total == 0:
        return 0

    p = values / total
    p = p[p > 0]

    return -np.sum(p * np.log2(p))

### Calcolo entropia

In [14]:
pivot["entropy"] = pivot.apply(shannon_entropy, axis=1)

pivot.head()

genres,userId,year,Action,Adventure,Animation,Children,Comedy,Crime,Documentary,Drama,...,Horror,IMAX,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western,entropy
0,1,2006,4.0,11.0,2.0,3.0,23.0,8.0,1.0,53.0,...,1.0,0.0,5.0,4.0,18.0,5.0,5.0,5.0,1.0,3.240401
1,2,2006,66.0,75.0,17.0,25.0,63.0,18.0,0.0,91.0,...,3.0,6.0,11.0,8.0,34.0,28.0,30.0,15.0,5.0,3.606494
2,3,2015,204.0,129.0,29.0,29.0,106.0,67.0,3.0,143.0,...,33.0,45.0,5.0,36.0,48.0,145.0,153.0,12.0,4.0,3.626402
3,3,2016,15.0,9.0,2.0,2.0,7.0,6.0,0.0,11.0,...,0.0,5.0,0.0,2.0,2.0,7.0,10.0,3.0,0.0,3.585458
4,3,2017,34.0,18.0,6.0,2.0,12.0,14.0,0.0,24.0,...,5.0,13.0,0.0,10.0,1.0,42.0,25.0,6.0,1.0,3.515761


### Controllo distribuzione

In [15]:
pivot["entropy"].describe()

count    240365.000000
mean          3.369608
std           0.453235
min          -0.000000
25%           3.294206
50%           3.493183
75%           3.619690
max           3.956198
Name: entropy, dtype: float64

In [16]:
ratings_year = (
    df.groupby(["userId", "year"])
    .size()
    .reset_index(name="n_ratings_year")
)

ratings_year.head()

,userId,year,n_ratings_year
0,1,2006,155
1,2,2006,524
2,3,2015,1239
3,3,2016,89
4,3,2017,220


### Creazione dataset finale

In [17]:
final_df = pivot.merge(
    user_year_novelty,
    on=["userId", "year"],
    how="left"
)

final_df = final_df.merge(
    ratings_year,
    on=["userId", "year"],
    how="left"
)

### Filtro sui campioni: prima 2 anni attivi, poi almneo 5 rating

In [18]:

# definisco i 19 generi ufficiali
GENRE_COLS = [
    "Action", "Adventure", "Animation", "Children", "Comedy", "Crime",
    "Documentary", "Drama", "Fantasy", "Film-Noir", "Horror", "IMAX",
    "Musical", "Mystery", "Romance", "Sci-Fi", "Thriller", "War", "Western"
]

print("Dataset iniziale prima dei filtri:", final_df.shape)

# 1. calcolo il rating totale basato sulle 19 colonne dei generi 
final_df["total_ratings"] = final_df[GENRE_COLS].sum(axis=1)

# 2. filtro le righe con meno di 5 recensioni/anno sui generi validi
final_df = final_df[final_df["total_ratings"] >= 5]
print("Shape dopo il filtro rating annuale (>= 5):", final_df.shape)

# 3. conto gli anni attivi DOPO il filtro dei rating
active_years = final_df.groupby("userId")["year"].transform("count")

# 4. filtro gli utenti con almeno 2 anni attivi
final_df = final_df[active_years >= 2].copy()

print("\n--- DATASET FINALE DOPO I FILTRI COMBINATI ---")
print("Shape finale del dataset:", final_df.shape)
print("Numero utenti longitudinali rimasti:", final_df["userId"].nunique())

Dataset iniziale prima dei filtri: (240365, 24)
Shape dopo il filtro rating annuale (>= 5): (235367, 25)

--- DATASET FINALE DOPO I FILTRI COMBINATI ---
Shape finale del dataset: (101722, 25)
Numero utenti longitudinali rimasti: 28896


### Introduzione variabile active_year

In [19]:
#ordino i dati per utente e anno prima di calcolare la sequenza
final_df = final_df.sort_values(["userId", "year"])

#creo la sequenza temporale dell'utente (active_year: 1, 2, 3...)
final_df["active_year"] = final_df.groupby("userId").cumcount() + 1

### Salvataggio dataset finale

In [20]:

output_path = data_processed / "user_year_genre_longitudinal.parquet"

final_df.to_parquet(output_path, index=False)

print("Dataset salvato in:", output_path)

Dataset salvato in: /Users/sofiadimaggio/Desktop/uni/3 anno/Tirocinio_MovieLens/data/processed/user_year_genre_longitudinal.parquet


### Controllo finale

In [21]:
df_check = pd.read_parquet(output_path)

print("Dimensioni del file salvato: ", df_check.shape)
df_check.head()

Dimensioni del file salvato:  (101722, 26)


,userId,year,Action,Adventure,Animation,Children,Comedy,Crime,Documentary,Drama,...,Romance,Sci-Fi,Thriller,War,Western,entropy,novelty_pct,n_ratings_year,total_ratings,active_year
0,3,2015,204.0,129.0,29.0,29.0,106.0,67.0,3.0,143.0,...,48.0,145.0,153.0,12.0,4.0,3.626402,1.0,1239,1239.0,1
1,3,2016,15.0,9.0,2.0,2.0,7.0,6.0,0.0,11.0,...,2.0,7.0,10.0,3.0,0.0,3.585458,0.0,89,89.0,2
2,3,2017,34.0,18.0,6.0,2.0,12.0,14.0,0.0,24.0,...,1.0,42.0,25.0,6.0,1.0,3.515761,0.0,220,220.0,3
3,3,2019,81.0,42.0,13.0,15.0,51.0,45.0,0.0,54.0,...,9.0,30.0,51.0,5.0,3.0,3.598108,0.0,457,457.0,4
4,5,1996,5.0,9.0,3.0,4.0,21.0,8.0,0.0,23.0,...,7.0,3.0,12.0,2.0,4.0,3.608284,1.0,116,116.0,1
